# Lab Exercise: SQL Analysis with Polars

In this lab, you'll practice SQL queries using Polars' built-in SQL functionality. Complete each exercise by writing the appropriate SQL query.

In [ ]:
# Setup – Run this cell first
import polars as pl

# Load data
airlines = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airlines.csv')
airports = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_airports.csv')
flights = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_flights.csv', null_values='NA')
planes = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_planes.csv', null_values='NA')
weather = pl.read_csv('https://raw.githubusercontent.com/philhetzel/opan5510-class11/refs/heads/main/data/nyc_weather.csv', null_values='NA', infer_schema_length=1000)

flights = flights.with_columns(pl.col("time_hour").str.strptime(pl.Datetime))
weather = weather.with_columns(pl.col("time_hour").str.strptime(pl.Datetime))

# Create SQL context
ctx = pl.SQLContext(
    airlines=airlines,
    airports=airports,
    flights=flights,
    planes=planes,
    weather=weather,
    eager_execution=True
)

print("Setup complete! Tables available:")
print(ctx.execute("SHOW TABLES"))


Setup complete! Tables available:
shape: (5, 1)
┌──────────┐
│ name     │
│ ---      │
│ str      │
╞══════════╡
│ airlines │
│ airports │
│ flights  │
│ planes   │
│ weather  │
└──────────┘


/tmp/ipython-input-968005105.py:15: DeprecationWarning: The argument `eager_execution` for `SQLContext.__init__` is deprecated. It has been renamed to `eager`.
  ctx = pl.SQLContext(


## Exercise 1: Basic Queries

### 1.1 Find all unique carriers in the airlines table

In [ ]:
# Write your SQL query here
result = ctx.execute("""
SELECT DISTINCT name FROM airlines""")

print(result)


shape: (16, 1)
┌────────────────────────┐
│ name                   │
│ ---                    │
│ str                    │
╞════════════════════════╡
│ Endeavor Air Inc.      │
│ American Airlines Inc. │
│ Alaska Airlines Inc.   │
│ JetBlue Airways        │
│ Delta Air Lines Inc.   │
│ …                      │
│ United Air Lines Inc.  │
│ US Airways Inc.        │
│ Virgin America         │
│ Southwest Airlines Co. │
│ Mesa Airlines Inc.     │
└────────────────────────┘


### 1.2 Find the top 10 destinations by number of flights

In [ ]:
# Write your SQL query here
result = ctx.execute("""
SELECT dest, COUNT(*) AS num_flights
FROM flights
GROUP BY dest
ORDER BY num_flights DESC LIMIT 10
""")

print(result)






shape: (10, 2)
┌──────┬─────────────┐
│ dest ┆ num_flights │
│ ---  ┆ ---         │
│ str  ┆ u32         │
╞══════╪═════════════╡
│ ORD  ┆ 17283       │
│ ATL  ┆ 17215       │
│ LAX  ┆ 16174       │
│ BOS  ┆ 15508       │
│ MCO  ┆ 14082       │
│ CLT  ┆ 14064       │
│ SFO  ┆ 13331       │
│ FLL  ┆ 12055       │
│ MIA  ┆ 11728       │
│ DCA  ┆ 9705        │
└──────┴─────────────┘


### 1.3 Find all flights that departed more than 2 hours late (120 minutes)

In [ ]:
# Write your SQL query here
result = ctx.execute("""
-- Your query here
""")

# print(result)

## Exercise 2: Aggregation

### 2.1 Calculate the average departure delay for each origin airport

In [ ]:
 # Write your SQL query here
result = ctx.execute("""
SELECT origin, AVG(dep_delay) AS avg_dep_delay
FROM flights
GROUP BY origin
""")

print(result)




shape: (3, 2)
┌────────┬───────────────┐
│ origin ┆ avg_dep_delay │
│ ---    ┆ ---           │
│ str    ┆ f64           │
╞════════╪═══════════════╡
│ LGA    ┆ 10.346876     │
│ EWR    ┆ 15.107954     │
│ JFK    ┆ 12.112159     │
└────────┴───────────────┘


### 2.2 Find the busiest month of the year

Count the number of flights per month and find which month has the most flights.

In [ ]:
# First, let's check what columns are available

result = ctx.execute("""
SELECT *
FROM flights
LIMIT 5
""")

print(result)

# Now write your query to find busiest month
result = ctx.execute("""
SELECT month, COUNT(*) AS num_flights FROM flights GROUP BY month
ORDER BY num_flights DESC LIMIT 1
""")

print(result)


shape: (5, 19)
┌──────┬───────┬─────┬──────────┬───┬──────────┬──────┬────────┬─────────────────────────┐
│ year ┆ month ┆ day ┆ dep_time ┆ … ┆ distance ┆ hour ┆ minute ┆ time_hour               │
│ ---  ┆ ---   ┆ --- ┆ ---      ┆   ┆ ---      ┆ ---  ┆ ---    ┆ ---                     │
│ i64  ┆ i64   ┆ i64 ┆ i64      ┆   ┆ i64      ┆ i64  ┆ i64    ┆ datetime[μs, UTC]       │
╞══════╪═══════╪═════╪══════════╪═══╪══════════╪══════╪════════╪═════════════════════════╡
│ 2013 ┆ 1     ┆ 1   ┆ 517      ┆ … ┆ 1400     ┆ 5    ┆ 15     ┆ 2013-01-01 10:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 533      ┆ … ┆ 1416     ┆ 5    ┆ 29     ┆ 2013-01-01 10:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 542      ┆ … ┆ 1089     ┆ 5    ┆ 40     ┆ 2013-01-01 10:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 544      ┆ … ┆ 1576     ┆ 5    ┆ 45     ┆ 2013-01-01 10:00:00 UTC │
│ 2013 ┆ 1     ┆ 1   ┆ 554      ┆ … ┆ 762      ┆ 6    ┆ 0      ┆ 2013-01-01 11:00:00 UTC │
└──────┴───────┴─────┴──────────┴───┴──────────┴──────┴────────┴───────────

### 2.3 Calculate the on-time performance rate for each carrier

Consider a flight on-time if the departure delay is <= 15 minutes.

In [ ]:
# Write your SQL query here
result = ctx.execute("""
SELECT carrier,
       COUNT(CASE WHEN dep_delay <= 15 THEN 1 END) * 1.0 / COUNT(*) AS on_time_rate
FROM flights
GROUP BY carrier
""")

print(result)


shape: (16, 2)
┌─────────┬──────────────┐
│ carrier ┆ on_time_rate │
│ ---     ┆ ---          │
│ str     ┆ f64          │
╞═════════╪══════════════╡
│ VX      ┆ 0.821        │
│ MQ      ┆ 0.751714     │
│ YV      ┆ 0.647255     │
│ US      ┆ 0.849873     │
│ AS      ┆ 0.865546     │
│ …       ┆ …            │
│ AA      ┆ 0.824376     │
│ B6      ┆ 0.76537      │
│ UA      ┆ 0.780431     │
│ 9E      ┆ 0.701246     │
│ FL      ┆ 0.716871     │
└─────────┴──────────────┘


## Exercise 3: Joins

### 3.1 List all flights with their airline names (not just carrier codes)

Show the first 20 flights with carrier code, airline name, flight number, origin, and destination.

In [ ]:
# Write your SQL query here
result = ctx.execute("""

SELECT f.carrier,a.name AS airline_name, f.flight, f.origin,f.dest

FROM flights f
JOIN airlines a ON f.carrier = a.carrier LIMIT 20
""")

print(result)



shape: (20, 5)
┌─────────┬────────────────────────┬────────┬────────┬──────┐
│ carrier ┆ airline_name           ┆ flight ┆ origin ┆ dest │
│ ---     ┆ ---                    ┆ ---    ┆ ---    ┆ ---  │
│ str     ┆ str                    ┆ i64    ┆ str    ┆ str  │
╞═════════╪════════════════════════╪════════╪════════╪══════╡
│ UA      ┆ United Air Lines Inc.  ┆ 1545   ┆ EWR    ┆ IAH  │
│ UA      ┆ United Air Lines Inc.  ┆ 1714   ┆ LGA    ┆ IAH  │
│ AA      ┆ American Airlines Inc. ┆ 1141   ┆ JFK    ┆ MIA  │
│ B6      ┆ JetBlue Airways        ┆ 725    ┆ JFK    ┆ BQN  │
│ DL      ┆ Delta Air Lines Inc.   ┆ 461    ┆ LGA    ┆ ATL  │
│ …       ┆ …                      ┆ …      ┆ …      ┆ …    │
│ B6      ┆ JetBlue Airways        ┆ 1806   ┆ JFK    ┆ BOS  │
│ UA      ┆ United Air Lines Inc.  ┆ 1187   ┆ EWR    ┆ LAS  │
│ B6      ┆ JetBlue Airways        ┆ 371    ┆ LGA    ┆ FLL  │
│ MQ      ┆ Envoy Air              ┆ 4650   ┆ LGA    ┆ ATL  │
│ B6      ┆ JetBlue Airways        ┆ 343    ┆ EWR    ┆ 

### 3.2 Find the average age of planes for each carrier

Hint: The planes table has a `year` column for manufacture year. Calculate age based on 2013.

In [ ]:
# Write your SQL query here
result = ctx.execute("""
SELECT f.carrier,
       AVG(2013 - p.year) AS avg_age FROM flights f
JOIN planes p ON f.tailnum = p.tailnum
  GROUP BY f.carrier
""")

print(result)

shape: (16, 2)
┌─────────┬───────────┐
│ carrier ┆ avg_age   │
│ ---     ┆ ---       │
│ str     ┆ f64       │
╞═════════╪═══════════╡
│ AS      ┆ 3.33662   │
│ F9      ┆ 4.87874   │
│ DL      ┆ 16.372169 │
│ HA      ┆ 1.548387  │
│ YV      ┆ 9.313758  │
│ …       ┆ …         │
│ FL      ┆ 11.385829 │
│ US      ┆ 9.103663  │
│ B6      ┆ 6.686702  │
│ UA      ┆ 13.207691 │
│ VX      ┆ 4.473643  │
└─────────┴───────────┘


### 3.3 Find flights that experienced both departure delays and bad weather

Join flights with weather data and find flights where departure delay > 30 minutes and either wind_speed > 20 or precip > 0.1

In [ ]:
# First, explore the weather table structure
result = ctx.execute("""
    SELECT *
    FROM weather
    LIMIT 5
""")
# print(result)

# Now write your join query
result = ctx.execute("""
    SELECT f.flight, f.origin, f.dest, f.dep_delay,
           w.wind_speed, w.precip
    FROM flights f
    JOIN weather w
      ON f.origin = w.origin
     AND f.year = w.year
     AND f.month = w.month
     AND f.day = w.day
     AND f.hour = w.hour
    WHERE f.dep_delay > 30
      AND (w.wind_speed > 20 OR w.precip > 0.1)
""")

print(result)


shape: (4_938, 6)
┌────────┬────────┬──────┬───────────┬────────────┬────────┐
│ flight ┆ origin ┆ dest ┆ dep_delay ┆ wind_speed ┆ precip │
│ ---    ┆ ---    ┆ ---  ┆ ---       ┆ ---        ┆ ---    │
│ i64    ┆ str    ┆ str  ┆ i64       ┆ f64        ┆ f64    │
╞════════╪════════╪══════╪═══════════╪════════════╪════════╡
│ 21     ┆ JFK    ┆ TPA  ┆ 47        ┆ 21.86482   ┆ 0.0    │
│ 199    ┆ JFK    ┆ LAS  ┆ 116       ┆ 21.86482   ┆ 0.0    │
│ 4090   ┆ EWR    ┆ JAX  ┆ 39        ┆ 20.71404   ┆ 0.0    │
│ 4231   ┆ EWR    ┆ IAD  ┆ 40        ┆ 24.16638   ┆ 0.0    │
│ 1010   ┆ JFK    ┆ BOS  ┆ 33        ┆ 20.71404   ┆ 0.0    │
│ …      ┆ …      ┆ …    ┆ …         ┆ …          ┆ …      │
│ 2347   ┆ LGA    ┆ ATL  ┆ 41        ┆ 20.71404   ┆ 0.0    │
│ 5503   ┆ LGA    ┆ PIT  ┆ 97        ┆ 20.71404   ┆ 0.0    │
│ 135    ┆ JFK    ┆ PHX  ┆ 42        ┆ 21.86482   ┆ 0.0    │
│ 797    ┆ JFK    ┆ LAX  ┆ 46        ┆ 21.86482   ┆ 0.0    │
│ 985    ┆ JFK    ┆ BOS  ┆ 45        ┆ 21.86482   ┆ 0.0    │
└─────

## Exercise 4: Advanced Queries

### 4.1 Find the most popular aircraft types (by number of flights)

Join flights with planes to get manufacturer and model information. Show top 10.

In [ ]:
# Write your SQL query here
result = ctx.execute("""
-- Your query here
SELECT p.manufacturer,
       p.model,
       COUNT(*) AS num_flights
FROM flights f
JOIN planes p
  ON f.tailnum = p.tailnum
GROUP BY p.manufacturer, p.model
ORDER BY num_flights DESC
LIMIT 10
""")

print(result)


shape: (10, 3)
┌───────────────────────────────┬─────────────────┬─────────────┐
│ manufacturer                  ┆ model           ┆ num_flights │
│ ---                           ┆ ---             ┆ ---         │
│ str                           ┆ str             ┆ u32         │
╞═══════════════════════════════╪═════════════════╪═════════════╡
│ AIRBUS                        ┆ A320-232        ┆ 31278       │
│ EMBRAER                       ┆ EMB-145LR       ┆ 28027       │
│ EMBRAER                       ┆ ERJ 190-100 IGW ┆ 23716       │
│ AIRBUS INDUSTRIE              ┆ A320-232        ┆ 14553       │
│ EMBRAER                       ┆ EMB-145XR       ┆ 14051       │
│ BOEING                        ┆ 737-824         ┆ 13809       │
│ BOMBARDIER INC                ┆ CL-600-2D24     ┆ 11807       │
│ BOEING                        ┆ 737-7H4         ┆ 10389       │
│ BOEING                        ┆ 757-222         ┆ 9150        │
│ MCDONNELL DOUGLAS AIRCRAFT CO ┆ MD-88           ┆ 8932     

### 4.2 Analyze route performance

Find the top 10 routes (origin-destination pairs) with:
- Total number of flights
- Average departure delay
- Percentage of flights delayed more than 30 minutes

Include airport names, not just codes.

In [ ]:
result = ctx.execute("""
SELECT
    f.origin,a1_name AS origin_airport,
    f.dest, a2_name AS dest_airport, COUNT(*) AS total_flights,
    AVG(f.dep_delay) AS avg_dep_delay,100.0 * SUM(CASE WHEN f.dep_delay > 30 THEN 1 ELSE 0 END) / COUNT(*) AS pct_over_30min
FROM flights f JOIN (
    SELECT faa, name AS a1_name
    FROM airports
) a1 ON f.origin = a1.faa
JOIN (
    SELECT faa, name AS a2_name
    FROM airports
) a2 ON f.dest = a2.faa
GROUP BY f.origin, a1_name, f.dest, a2_name ORDER BY total_flights DESC
LIMIT 10
""")

print(result)

shape: (10, 7)
┌────────┬────────────────┬──────┬────────────────┬───────────────┬───────────────┬────────────────┐
│ origin ┆ origin_airport ┆ dest ┆ dest_airport   ┆ total_flights ┆ avg_dep_delay ┆ pct_over_30min │
│ ---    ┆ ---            ┆ ---  ┆ ---            ┆ ---           ┆ ---           ┆ ---            │
│ str    ┆ str            ┆ str  ┆ str            ┆ u32           ┆ f64           ┆ f64            │
╞════════╪════════════════╪══════╪════════════════╪═══════════════╪═══════════════╪════════════════╡
│ JFK    ┆ John F Kennedy ┆ LAX  ┆ Los Angeles    ┆ 11262         ┆ 8.522508      ┆ 9.829515       │
│        ┆ Intl           ┆      ┆ Intl           ┆               ┆               ┆                │
│ LGA    ┆ La Guardia     ┆ ATL  ┆ Hartsfield     ┆ 10263         ┆ 11.448621     ┆ 12.247881      │
│        ┆                ┆      ┆ Jackson        ┆               ┆               ┆                │
│        ┆                ┆      ┆ Atlanta Int…   ┆               ┆         

In [ ]:
# Write your SQL query here
result = ctx.execute("""
-- Your query here
""")

# print(result)

## Bonus: Compare with Polars

### Choose one of the queries above and implement it using Polars

This will help you understand the relationship between SQL and Polars operations.

In [ ]:
# Example: Let's implement Exercise 2.1 (average delay by origin) in Polars

# SQL version (for reference)
sql_result = ctx.execute("""
    SELECT
        origin,
        AVG(dep_delay) as avg_delay
    FROM flights
    WHERE dep_delay IS NOT NULL
    GROUP BY origin
    ORDER BY avg_delay DESC
""")

# Polars version
polars_result = (
    flights
    .filter(pl.col('dep_delay').is_not_null())
    .group_by('origin')
    .agg(pl.col('dep_delay').mean().alias('avg_delay'))
    .sort('avg_delay', descending=True)
)

print("SQL Result:")
print(sql_result)
print("\nPolars Result:")
print(polars_result)


# Polars version of route performance
polars_result = (
    flights
    .join(airports.rename({"faa": "origin"}).select(["origin", "name"]), on="origin")
    .rename({"name": "origin_airport"})
    .join(airports.rename({"faa": "dest"}).select(["dest", "name"]), on="dest")
    .rename({"name": "dest_airport"})
    .group_by(["origin", "origin_airport", "dest", "dest_airport"])
    .agg([
        pl.count().alias("total_flights"),
        pl.col("dep_delay").mean().alias("avg_dep_delay"),
        (100 * (pl.col("dep_delay") > 30).cast(pl.Int32).mean()).alias("pct_over_30min")
    ])
    .sort("total_flights", descending=True)
    .limit(10)
)

print(polars_result)


SQL Result:
shape: (3, 2)
┌────────┬───────────┐
│ origin ┆ avg_delay │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ EWR    ┆ 15.107954 │
│ JFK    ┆ 12.112159 │
│ LGA    ┆ 10.346876 │
└────────┴───────────┘

Polars Result:
shape: (3, 2)
┌────────┬───────────┐
│ origin ┆ avg_delay │
│ ---    ┆ ---       │
│ str    ┆ f64       │
╞════════╪═══════════╡
│ EWR    ┆ 15.107954 │
│ JFK    ┆ 12.112159 │
│ LGA    ┆ 10.346876 │
└────────┴───────────┘
shape: (10, 7)
┌────────┬────────────────┬──────┬────────────────┬───────────────┬───────────────┬────────────────┐
│ origin ┆ origin_airport ┆ dest ┆ dest_airport   ┆ total_flights ┆ avg_dep_delay ┆ pct_over_30min │
│ ---    ┆ ---            ┆ ---  ┆ ---            ┆ ---           ┆ ---           ┆ ---            │
│ str    ┆ str            ┆ str  ┆ str            ┆ u32           ┆ f64           ┆ f64            │
╞════════╪════════════════╪══════╪════════════════╪═══════════════╪═══════════════╪════════════════╡
│ JFK    ┆

/tmp/ipython-input-4177712576.py:38: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().alias("total_flights"),
